## Task 1 Base Task. 

Use Credit-g dataset to get TabPFN running end-to-end. Also run baselines (XGBoost, CatBoost, Logistic Regression, Random Forest). Compare the methods on ROC AUC, log loss, train latency and predict latency

## Set Up TabPFN client connection

In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"

Import package

In [ ]:
try:
    from tabpfn import TabPFNClassifier, TabPFNRegressor
except ImportError:
    raise ImportError(
        "Warning: Could not import TabPFN. Please run installation above and restart the session afterwards (Runtime > Restart Session)."
    )

# Data Science & Visualization
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import time

# Other ML Models
from catboost import CatBoostClassifier, CatBoostRegressor

# Notebook UI/Display
from IPython.display import Markdown, display
from rich.console import Console
from rich.panel import Panel
from rich.rule import Rule
from sklearn.compose import make_column_selector, make_column_transformer

# Scikit-Learn: Data & Preprocessing
from sklearn.datasets import fetch_openml, load_breast_cancer

# Scikit-Learn: Models
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import mean_squared_error, roc_auc_score, log_loss, accuracy_score
from sklearn.model_selection import (
    KFold,
    StratifiedKFold,
    cross_val_score,
    cross_validate,
    train_test_split,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier, XGBRegressor

# This transformer will be used to handle categorical features for the baseline models
column_transformer = make_column_transformer(
    (
        OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1),
        make_column_selector(dtype_include=["object", "category"]),
    ),
    remainder="passthrough",
)

In [37]:
console = Console()

console.print(Panel.fit("[bold magenta]TabPFN Demo: Backend Selection[/bold magenta]"))
console.print("\nThis script can run TabPFN using one of two backends:")
console.print("  [bold]local:[/bold] Uses a local GPU (NVIDIA). Requires CUDA.")
console.print(
    "  [bold]client:[/bold] Uses the TabPFN API. Requires an internet connection and a free account."
)

backend = None
while backend is None:
  console.print(
      "\n[bold]Choose your backend[/bold]: - If no text box is shown, restart the cell.",
  )
  user_input = input("Enter 'local' or 'client' and press return:")
  if user_input not in ["local", "client"]:
    continue
  backend = user_input

console.print(
    f"\n✅ You have selected the '[bold green]{backend}[/bold green]' backend."
)

console.print(Rule(f"[bold]Setting up [cyan]{backend}[/cyan] backend[/bold]"))

if backend == "local":
    console.print("Attempting local backend setup...")
    import torch

    if not torch.cuda.is_available():
        console.print(
            "[bold red]Error:[/bold red] GPU device not found. For fast training, please enable GPU.",
            style="red",
        )
        console.print(
            "In Colab: Go to [bold]Runtime -> Change runtime type -> Hardware accelerator -> GPU.[/bold]",
            style="yellow",
        )
        raise SystemError("GPU device not found.")
    console.print("[bold green]✅ GPU is available.[/bold green]")

    # --- Prior Labs Authentication ---
    console.print(Rule("[bold]Prior Labs Authentication[/bold]"))
    console.print(
        "\nTabPFN model weights require a free [bold]Prior Labs account[/bold] and "
        "acceptance of the non-commercial license.\n"
    )

    import os
    import getpass

    tabpfn_token = None

    # 1. Try Colab secret TABPFN_TOKEN
    try:
        from google.colab import userdata
        tabpfn_token = userdata.get("TABPFN_TOKEN")
        if tabpfn_token:
            os.environ["TABPFN_TOKEN"] = tabpfn_token
            console.print("[bold green]✅ Found TABPFN_TOKEN in Colab secrets.[/bold green]")
    except Exception:
        pass

    # 2. If no token found, prompt the user
    if not tabpfn_token:
        console.print(
            Panel(
                "To get your access token:\n\n"
                "  1. Go to [link=https://ux.priorlabs.ai]ux.priorlabs.ai[/link] and sign up / log in\n"
                "  2. Accept the license at [link=https://ux.priorlabs.ai/account/licenses]ux.priorlabs.ai/account/licenses[/link]\n"
                "  3. Copy your Access Token from [link=https://ux.priorlabs.ai/account]ux.priorlabs.ai/account[/link]\n\n"
                "[bold yellow]Tip:[/bold yellow] Save the token as a Colab secret named "
                "[bold cyan]TABPFN_TOKEN[/bold cyan] to skip this step next time.",
                title="[bold]🔑 Prior Labs Access Token required",
                border_style="blue",
            )
        )
        while not tabpfn_token:
            token_input = getpass.getpass("Paste your TABPFN_TOKEN and press Enter: ")
            if token_input.strip():
                tabpfn_token = token_input.strip()
                os.environ["TABPFN_TOKEN"] = tabpfn_token
            else:
                console.print("[red]Token cannot be empty. Please try again.[/red]")

    console.print("")
    console.print("Importing local TabPFN library...")

    from tabpfn import TabPFNClassifier, TabPFNRegressor

    console.print("[bold green]✅ TabPFN (local) imported successfully.[/bold green]")
elif backend == "client":
    console.print("Attempting client backend setup...")
    console.print("Importing TabPFN client library...")
    from tabpfn_client import TabPFNClassifier, TabPFNRegressor, init

    init()
    console.print("[bold green]✅ TabPFN (client) initialized.[/bold green]")

╭────────────────────────────────╮
│ TabPFN Demo: Backend Selection │
╰────────────────────────────────╯

This script can run TabPFN using one of two backends:

local: Uses a local GPU (NVIDIA). Requires CUDA.

client: Uses the TabPFN API. Requires an internet connection and a free account.

Choose your backend: - If no text box is shown, restart the cell.

✅ You have selected the 'client' backend.

──────────────────────────────────────────── Setting up client backend ────────────────────────────────────────────

Attempting client backend setup...

Importing TabPFN client library...

✅ TabPFN (client) initialized.

Load credit-g data

In [38]:
credit = fetch_openml('credit-g', version = 1, as_frame = True)
X = credit.data
y = credit.target 

display(X)

,checking_status,duration,credit_history,purpose,credit_amount,savings_status,employment,installment_commitment,personal_status,other_parties,residence_since,property_magnitude,age,other_payment_plans,housing,existing_credits,job,num_dependents,own_telephone,foreign_worker
0,<0,6,critical/other existing credit,radio/tv,1169,no known savings,>=7,4,male single,none,4,real estate,67,none,own,2,skilled,1,yes,yes
1,0<=X<200,48,existing paid,radio/tv,5951,<100,1<=X<4,2,female div/dep/mar,none,2,real estate,22,none,own,1,skilled,1,none,yes
2,no checking,12,critical/other existing credit,education,2096,<100,4<=X<7,2,male single,none,3,real estate,49,none,own,1,unskilled resident,2,none,yes
3,<0,42,existing paid,furniture/equipment,7882,<100,4<=X<7,2,male single,guarantor,4,life insurance,45,none,for free,1,skilled,2,none,yes
4,<0,24,delayed previously,new car,4870,<100,1<=X<4,3,male single,none,4,no known property,53,none,for free,2,skilled,2,none,yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,no checking,12,existing paid,furniture/equipment,1736,<100,4<=X<7,3,female div/dep/mar,none,4,real estate,31,none,own,1,unskilled resident,1,none,yes
996,<0,30,existing paid,used car,3857,<100,1<=X<4,4,male div/sep,none,4,life insurance,40,none,own,1,high qualif/self emp/mgmt,1,yes,yes
997,no checking,12,existing paid,radio/tv,804,<100,>=7,4,male single,none,4,car,38,none,own,1,skilled,1,none,yes
998,<0,45,existing paid,radio/tv,1845,<100,1<=X<4,4,male single,none,4,no known property,23,none,for free,1,skilled,1,yes,yes


In [39]:
# processing categorical data
categorical_columns = X.select_dtypes(include=['object', "category"]).columns

oe = OrdinalEncoder()
encoded_data = oe.fit_transform(X[categorical_columns])
encoded_X = pd.DataFrame(
    encoded_data,
    columns = categorical_columns,
    index = X.index
)

final_X = pd.concat(
    [X.drop(columns=categorical_columns), encoded_X],
    axis = 1
)

le = LabelEncoder()
y = le.fit_transform(y)

display(final_X)

,duration,credit_amount,installment_commitment,residence_since,age,existing_credits,num_dependents,checking_status,credit_history,purpose,savings_status,employment,personal_status,other_parties,property_magnitude,other_payment_plans,housing,job,own_telephone,foreign_worker
0,6,1169,4,4,67,2,1,1.0,1.0,6.0,4.0,3.0,3.0,2.0,3.0,1.0,1.0,1.0,1.0,1.0
1,48,5951,2,2,22,1,1,0.0,3.0,6.0,2.0,0.0,0.0,2.0,3.0,1.0,1.0,1.0,0.0,1.0
2,12,2096,2,3,49,1,2,3.0,1.0,2.0,2.0,1.0,3.0,2.0,3.0,1.0,1.0,3.0,0.0,1.0
3,42,7882,2,4,45,1,2,1.0,3.0,3.0,2.0,1.0,3.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0
4,24,4870,3,4,53,2,2,1.0,2.0,4.0,2.0,0.0,3.0,2.0,2.0,1.0,0.0,1.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,12,1736,3,4,31,1,1,3.0,3.0,3.0,2.0,1.0,0.0,2.0,3.0,1.0,1.0,3.0,0.0,1.0
996,30,3857,4,4,40,1,1,1.0,3.0,9.0,2.0,0.0,1.0,2.0,1.0,1.0,1.0,0.0,1.0,1.0
997,12,804,4,4,38,1,1,3.0,3.0,6.0,2.0,3.0,3.0,2.0,0.0,1.0,1.0,1.0,0.0,1.0
998,45,1845,4,4,23,1,1,1.0,3.0,6.0,2.0,0.0,3.0,2.0,2.0,1.0,0.0,1.0,1.0,1.0


<medium> Run Tab-PFN end-to-end (fit + predict) and get relevant metrics. </medium>

In [40]:
X_train, X_test, y_train, y_test = train_test_split(
    final_X, y, test_size=0.20, random_state=42
)

# Train and evaluate the TabPFN classifier
tabpfn_classifier = TabPFNClassifier(random_state=42)

train_start = time.perf_counter()
tabpfn_classifier.fit(X_train, y_train)
train_end = time.perf_counter()

predict_start = time.perf_counter()
y_pred_proba_tabpfn = tabpfn_classifier.predict_proba(X_test)
predict_end = time.perf_counter()

train_latency = train_end - train_start
predict_latency = predict_end - predict_start
end_to_end_latency = predict_end - train_start

roc_auc_tabpfn = roc_auc_score(y_test, y_pred_proba_tabpfn[:, 1])
log_loss_tabpfn = log_loss(y_test, y_pred_proba_tabpfn)

print(f"TabPFN ROC AUC Score: {roc_auc_tabpfn:.4f}")
print(f"TabPFN Log Loss: {log_loss_tabpfn:.4f}")
print(f"TabPFN Train Latency: {train_latency:.4f} seconds")
print(f"TabPFN Predict Latency: {predict_latency:.4f} seconds")
print(f"TabPFN End-to-End Latency: {end_to_end_latency:.4f} seconds")

00:00 Fitting... -

The provided train set hashes match previously uploaded train sets.


00:06 Fitting... Done!
00:00 Predicting... \

The provided test set hash matches a previously uploaded test set.


00:02 Predicting... Done!
TabPFN ROC AUC Score: 0.8340
TabPFN Log Loss: 0.4478
TabPFN Train Latency: 6.5258 seconds
TabPFN Predict Latency: 2.6632 seconds
TabPFN End-to-End Latency: 9.1892 seconds


/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:3001: UserWarning: The y_pred values do not sum to one. Make sure to pass probabilities.
  warnings.warn(


In [52]:
models = [
    ("TabPFN", TabPFNClassifier(random_state=42)),
    (
        "RandomForest",
        make_pipeline(
            column_transformer,  # string data needs to be encoded for model
            RandomForestClassifier(random_state=42),
        ),
    ),
    (
        "XGBoost",
        make_pipeline(
            column_transformer,  # string data needs to be encoded for model
            XGBClassifier(random_state=42),
        ),
    ),
    (
        "CatBoost",
        make_pipeline(
            column_transformer,  # string data needs to be encoded for model
            CatBoostClassifier(random_state=42, verbose=0),
        ),
    ),
    (
        "Logistic Regression",
        make_pipeline(
            column_transformer,
            LogisticRegression(random_state=42, max_iter = 5000, solver="saga")
        )
    )
]

n_splits = 2
cv = StratifiedKFold(n_splits=n_splits, random_state=42, shuffle=True)
scoring = ["neg_log_loss", "roc_auc"]
cv_results = {
    name: cross_validate(model, final_X, y, cv=cv, scoring=scoring, n_jobs=1, verbose=1, return_train_score=False)
    for name, model in models
}

print(cv_results)

# sklearn returns negative log loss because higher scores are considered better.
df = pd.DataFrame(
    [
        {
            "Model": name,
            "ROC AUC": results["test_roc_auc"].mean(),
            "Log Loss Mean": -results["test_neg_log_loss"].mean(),
            "Log Loss Std": results["test_neg_log_loss"].std(),
            "Train Latency Mean (s)": results["fit_time"].mean(),
            "Predict/Score Latency Mean (s)": results["score_time"].mean(),
            "End-to-End CV Latency (s)": results["fit_time"].sum() + results["score_time"].sum(),
        }
        for name, results in cv_results.items()
    ]
)

df


00:00 Fitting... -

The provided train set hashes match previously uploaded train sets.


00:06 Fitting... Done!
00:00 Predicting... \

The provided test set hash matches a previously uploaded test set.


00:02 Predicting... Done!
00:00 Fitting... \

The provided train set hashes match previously uploaded train sets.


00:05 Fitting... Done!
00:00 Predicting... \

The provided test set hash matches a previously uploaded test set.


00:02 Predicting... Done!
{'TabPFN': {'fit_time': array([6.32330704, 5.80360699]), 'score_time': array([2.6705451 , 2.67200637]), 'test_neg_log_loss': array([-0.51588764, -0.4978665 ]), 'test_roc_auc': array([0.7740381 , 0.78460952])}, 'RandomForest': {'fit_time': array([0.22750401, 0.27752304]), 'score_time': array([0.01722598, 0.01804185]), 'test_neg_log_loss': array([-0.50815374, -0.51196242]), 'test_roc_auc': array([0.77378095, 0.76666667])}, 'XGBoost': {'fit_time': array([0.03533912, 0.04309869]), 'score_time': array([0.00719619, 0.00975704]), 'test_neg_log_loss': array([-0.70391645, -0.67135772]), 'test_roc_auc': array([0.75108571, 0.75939048])}, 'CatBoost': {'fit_time': array([1.05428076, 0.73931003]), 'score_time': array([0.01726842, 0.00390792]), 'test_neg_log_loss': array([-0.50481241, -0.49204754]), 'test_roc_auc': array([0.77695238, 0.7948381 ])}, 'Logistic Regression': {'fit_time': array([0.249897  , 0.27069378]), 'score_time': array([0.00363803, 0.00398421]), 'test_neg_lo

,Model,ROC AUC,Log Loss Mean,Log Loss Std,Train Latency Mean (s),Predict/Score Latency Mean (s),End-to-End CV Latency (s)
0,TabPFN,0.779324,0.506877,0.009011,6.063457,2.671276,17.469465
1,RandomForest,0.770224,0.510058,0.001904,0.252514,0.017634,0.540295
2,XGBoost,0.755238,0.687637,0.016279,0.039219,0.008477,0.095391
3,CatBoost,0.785895,0.498430,0.006382,0.896795,0.010588,1.814767
4,Logistic Regression,0.594819,0.598138,0.003594,0.260295,0.003811,0.528213
